In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import torchvision.models as models
import torch.nn as nn

from src.MyDataset import MyDataset
from configs.config import *

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
test_set = MyDataset(TEST_CSV, TEST_IMG_DIR, transform=transform)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

In [ ]:
classes = ("apple", "banana", "orange", "pineapple", "watermelon")

In [ ]:
net = models.resnet18(weights=None)

net.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(512,5)
)

net.load_state_dict(
    torch.load(MODEL_PATH, weights_only=True)
)

net.eval()

In [ ]:
correct = 0
total = 0

all_labels = []
all_outputs = []

net.eval()

with torch.no_grad():

    for images, labels in test_loader:

        outputs = net(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy().tolist())
        all_outputs.extend(predicted.cpu().numpy().tolist())

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


print(
    f'Accuracy of the network on the {total} test images: '
    f'{100 * correct / total:.2f}%'
)

In [ ]:
accuracy = 100 * correct / total

experiment_path = f"./logs/{MODEL_NAME}.txt"

with open(experiment_path, "a") as f:
    f.write(
        f"\n\nFinal Test Accuracy:\n{accuracy:.2f}%"
    )

print("Accuracy added to experiment file")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Make sure labels are normal integers
all_labels = [int(x) for x in all_labels]
all_outputs = [int(x) for x in all_outputs]

# Create confusion matrix
cm = confusion_matrix(
    all_labels,
    all_outputs
)

print(cm)

# Display confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

disp.plot(
    cmap="Blues",
    values_format="d"
)

plt.title("Transfer Learning Model Confusion Matrix using ResNet18")

plt.savefig(
    f"./results/confusion_matrices/{MODEL_NAME}_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()